# Deepfake Detection — DFNet training on Kaggle GPU

**Before you run anything**, open the panel on the right and set:

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 x2** (or P100) |
| Internet | **On** (needed for `git clone`) |
| Input | Add dataset **`xhlulu/140k-real-and-fake-faces`** |

Then `Run All`. A full 25-epoch run takes roughly 2–3 hours, well inside
Kaggle's 9-hour GPU session limit and its 30 GPU-hours/week quota.

## 1. Confirm the GPU is actually attached

In [ ]:
import shutil, subprocess, torch

# shutil.which first: subprocess.run raises FileNotFoundError if the binary is
# absent, which is exactly the no-GPU case we are trying to report cleanly.
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                          "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip())
else:
    print("nvidia-smi not found - this session has no GPU attached")

print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

assert torch.cuda.is_available(), (
    "No GPU attached.\n"
    "Fix: click 'Edit' to open the notebook editor, then in the right-hand\n"
    "sidebar set Session options -> Accelerator -> 'GPU T4 x2'.\n"
    "The session restarts; then Run All again.")

## 2. Get the project code

Cloned fresh from GitHub every run, so **push your local changes first** or
Kaggle will train an older version:

```bash
# on your Mac, before re-running this notebook
git add -A && git commit -m "your message" && git push
```

The repo must stay **public** for this clone to work without credentials.

In [ ]:
REPO_URL = "https://github.com/Preet1002/Deepfake_Detection.git"

import os, shutil, subprocess, sys
from pathlib import Path

PROJECT = Path("/kaggle/working/Deepfake_Detection")
if PROJECT.exists():
    shutil.rmtree(PROJECT)          # always start from a clean clone

subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT)], check=True)

os.chdir(PROJECT)                   # `!python` cells below inherit this cwd
sys.path.insert(0, str(PROJECT))
print("working dir:", os.getcwd())
print(sorted(p.name for p in PROJECT.iterdir() if not p.name.startswith(".")))

## 3. Locate the dataset mount

Kaggle mounts inputs read-only, so we point the loader straight at it. The
dataset ships `train/valid/test`; the loader accepts `valid` as the val split,
so **no copying or preparation is needed**.

This searches `/kaggle/input` for *any* folder shaped like split/real+fake, so
it works with re-uploaded mirrors of the dataset, not just one exact slug.

In [ ]:
from pathlib import Path

INPUT = Path("/kaggle/input")
SPLIT_NAMES = {"train", "training", "valid", "validation", "val", "test", "testing"}


def subdirs(path):
    try:
        return [c for c in path.iterdir() if c.is_dir()]
    except (PermissionError, OSError):
        return []


def usable_splits(path):
    """Split folders under `path` that actually contain both real/ and fake/."""
    return [c for c in subdirs(path)
            if c.name.lower() in SPLIT_NAMES
            and {d.name.lower() for d in subdirs(c)} >= {"real", "fake"}]


def is_dataset_root(path):
    return len(usable_splits(path)) >= 2


def find_roots(start, max_depth=8):
    """Breadth-first search, never descending into the image folders.

    Depth has to be generous: Kaggle nests some mounts as
    /kaggle/input/datasets/<owner>/<slug>/... which puts the real root six
    levels down. This stays cheap because we stop as soon as a directory looks
    like a dataset root and never call iterdir() inside real/ or fake/.
    """
    found, frontier = [], [(start, 0)]
    while frontier:
        path, depth = frontier.pop(0)
        if is_dataset_root(path):
            found.append(path)
            continue                       # no need to look deeper here
        if depth < max_depth:
            frontier += [(c, depth + 1) for c in subdirs(path)
                         if c.name.lower() not in {"real", "fake"}]
    return found


def print_tree(path, prefix="", depth=0, max_depth=5):
    for child in sorted(subdirs(path))[:12]:
        print(f"{prefix}  {child.name}/")
        if depth < max_depth:
            print_tree(child, prefix + "  ", depth + 1, max_depth)


roots = find_roots(INPUT)
if not roots:
    print("No dataset found. Directory tree under /kaggle/input:")
    print_tree(INPUT)
    raise SystemExit(
        "Add a real/fake face dataset via '+ Add Input'. Paste this URL into "
        "the search box: "
        "https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces")

# Prefer the copy with the most usable splits: a mirror with only train/test
# would leave training with no validation set to select checkpoints on.
roots.sort(key=lambda r: len(usable_splits(r)), reverse=True)
if len(roots) > 1:
    print("Multiple candidates found:")
    for r in roots:
        print(f"   {len(usable_splits(r))} splits  {r}")
    print("Using the first.\n")

DATA_ROOT = str(roots[0])
names = {c.name.lower() for c in usable_splits(Path(DATA_ROOT))}
print("dataset root:", DATA_ROOT)
for split in sorted(usable_splits(Path(DATA_ROOT))):
    counts = {c.name: sum(1 for _ in c.iterdir()) for c in sorted(subdirs(split))}
    print(f"  {split.name:11s} {counts}")

if not names & {"val", "valid", "validation"}:
    raise SystemExit(
        "\nThis copy has no validation split, which training needs to select "
        "the best checkpoint.\nUse the original dataset instead - paste this "
        "into '+ Add Input':\n"
        "  https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces")

## 4. Write the run config

Built from `configs/kaggle.yaml` with the detected dataset path patched in.

In [ ]:
import yaml
from pathlib import Path

config = yaml.safe_load(Path("configs/kaggle.yaml").read_text())
config["data"]["root"] = DATA_ROOT

Path("/kaggle/working/configs").mkdir(parents=True, exist_ok=True)

# A capped copy for the shakedown run, and the real one for the full run.
smoke = yaml.safe_load(yaml.safe_dump(config))
smoke["data"]["limit_per_class"] = 2000
smoke["train"].update(epochs=2, warmup_epochs=0,
                      out_dir="/kaggle/working/runs/smoke")
Path("/kaggle/working/configs/smoke.yaml").write_text(yaml.safe_dump(smoke))
Path("/kaggle/working/configs/full.yaml").write_text(yaml.safe_dump(config))

print(yaml.safe_dump(config, sort_keys=False))

## 5. Shakedown run (~4 minutes)

2 epochs on 2k images per class. The numbers are meaningless — this only proves
the CUDA path, the data mount and the checkpointing all work before you commit
to a multi-hour run. **The local development was verified on Apple MPS, so this
is the first real exercise of the CUDA + AMP path.**

In [ ]:
!python -u -m src.train --config /kaggle/working/configs/smoke.yaml

## 6. Full training run (~2–3 hours)

Watch the first epoch's timing. If it is much slower than ~6 min, the 4 vCPUs
are bottlenecking on JPEG augmentation rather than the GPU — lower `aug.jpeg`
to `0.3` in the config and re-run.

In [ ]:
!python -u -m src.train --config /kaggle/working/configs/full.yaml

## 7. Evaluate on the held-out test split

Scored in full, even though training used the capped subset in the smoke run.

In [ ]:
!python -u -m src.evaluate \
    --checkpoint /kaggle/working/runs/dfnet/best.pt \
    --split test \
    --batch-size 256

## 8. Training curves and test figures

In [ ]:
import json
import matplotlib.pyplot as plt
from pathlib import Path

history = json.loads(Path("/kaggle/working/runs/dfnet/history.json").read_text())
epochs = [h["epoch"] for h in history]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, [h["train_loss"] for h in history], label="train")
axes[0].plot(epochs, [h["val_loss"] for h in history], label="val")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("BCE loss"); axes[0].legend()
axes[0].set_title("Loss")

axes[1].plot(epochs, [h["auc"] for h in history], label="val AUC")
axes[1].plot(epochs, [h["accuracy"] for h in history], label="val accuracy")
axes[1].set_xlabel("epoch"); axes[1].legend(); axes[1].set_title("Validation")
plt.tight_layout(); plt.show()

best = max(history, key=lambda h: h["auc"])
print(f"best epoch {best['epoch']}: AUC={best['auc']:.4f} acc={best['accuracy']:.4f}")

In [ ]:
from IPython.display import Image as IPyImage, display

for name in ("roc.png", "confusion_matrix.png"):
    path = f"/kaggle/working/runs/dfnet/eval_test/{name}"
    if Path(path).exists():
        display(IPyImage(filename=path))

## 9. Grad-CAM on a few test images

Warm regions are what pushed the decision towards FAKE.

In [ ]:
import random
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

from src.predict import Detector

detector = Detector("/kaggle/working/runs/dfnet/best.pt")
random.seed(0)

samples = []
for class_name in ("real", "fake"):
    folder = Path(DATA_ROOT) / "test" / class_name
    samples += [(p, class_name) for p in random.sample(sorted(folder.glob("*.jpg")), 3)]

fig, axes = plt.subplots(2, len(samples), figsize=(3 * len(samples), 6.5))
for col, (path, truth) in enumerate(samples):
    image = Image.open(path).convert("RGB")
    # These are already tight face crops, so skip detection.
    result = detector.predict(image, detect_faces=False, explain=True)
    axes[0][col].imshow(image); axes[0][col].axis("off")
    axes[0][col].set_title(f"truth: {truth}", fontsize=10)
    axes[1][col].imshow(result.faces[0].heatmap_image); axes[1][col].axis("off")
    correct = result.label.lower() == truth
    axes[1][col].set_title(f"{result.label} p={result.fake_probability:.2f} "
                           f"{'OK' if correct else 'WRONG'}",
                           fontsize=10, color="green" if correct else "red")
plt.tight_layout(); plt.show()

## 10. Save the checkpoint

Everything under `/kaggle/working/` is kept as notebook output (20 GB limit).
Download `best.pt` and drop it into `runs/dfnet/` on your Mac — the checkpoint
carries its own config, so `evaluate.py` and the web app load it unchanged.

Delete the smoke run first so you are not shipping a useless 300 KB file
alongside the real one.

In [ ]:
import shutil
from pathlib import Path

shutil.rmtree("/kaggle/working/runs/smoke", ignore_errors=True)
shutil.rmtree("/kaggle/working/Deepfake_Detection", ignore_errors=True)  # re-cloneable

for p in sorted(Path("/kaggle/working").rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size / 1e6:8.2f} MB  {p}")